# Training Tiny Models for Mechanistic Interpretability

Tiny models trained from scratch on controlled tasks are invaluable for mechinterp research:
they're fast to train, fully understood, and exhibit phenomena like **grokking**.

This notebook covers the `irtk.training` module:
- Creating algorithmic datasets (modular arithmetic, copy, reverse, sort)
- Training small transformers from scratch
- Analyzing how model internals change across training checkpoints

## Why This Matters

Training small models from scratch lets you study how circuits form during learning. By controlling the training data and observing weight evolution, you can identify phase transitions (like the sudden emergence of induction heads) and test hypotheses about what drives circuit formation.

**Key references:**
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.training import create_algorithmic_dataset, train_tiny_model, training_checkpoint_analysis

## 1. Algorithmic Datasets

`create_algorithmic_dataset` generates labeled data for several controlled tasks.

In [ ]:
# Modular addition: a + b mod p
data_add = create_algorithmic_dataset("modular_addition", n_samples=200, modulus=13)
print(f"Tokens shape: {data_add['tokens'].shape}  (samples, [a, b, =])")
print(f"Labels shape: {data_add['labels'].shape}")
print(f"Example: {data_add['tokens'][0]} -> {data_add['labels'][0]}")
print(f"Check: {data_add['tokens'][0][0]} + {data_add['tokens'][0][1]} mod 13 = {(data_add['tokens'][0][0] + data_add['tokens'][0][1]) % 13}")

In [ ]:
# Modular subtraction: a - b mod p
data_sub = create_algorithmic_dataset("modular_subtraction", n_samples=100, modulus=7)
print(f"\nSubtraction example: {data_sub['tokens'][0]} -> {data_sub['labels'][0]}")

# Copy task: predict first token of sequence after separator
data_copy = create_algorithmic_dataset("copy", n_samples=100, modulus=10, seq_len=4)
print(f"Copy tokens shape: {data_copy['tokens'].shape}  (samples, [seq..., sep])")
print(f"Copy example: {data_copy['tokens'][0]} -> {data_copy['labels'][0]}")

# Sort task: predict minimum of input sequence
data_sort = create_algorithmic_dataset("sort", n_samples=100, modulus=10, seq_len=4)
print(f"Sort example: {data_sort['tokens'][0]} -> {data_sort['labels'][0]} (min of {data_sort['tokens'][0][:4]})")

## 2. Training a Tiny Transformer

Train a 1-layer model on modular addition. This is the classic **grokking** setup.

In [ ]:
# Config for a tiny model
cfg = irtk.HookedTransformerConfig(
    n_layers=1,
    d_model=32,
    n_ctx=4,
    d_head=16,
    n_heads=2,
    d_vocab=14,  # 0-12 for values + 13 for '='
)

# Create train/val split
data = create_algorithmic_dataset("modular_addition", n_samples=500, modulus=13)
n_train = 350
train_tokens, val_tokens = data["tokens"][:n_train], data["tokens"][n_train:]
train_labels, val_labels = data["labels"][:n_train], data["labels"][n_train:]

print(f"Train: {train_tokens.shape[0]} samples")
print(f"Val:   {val_tokens.shape[0]} samples")

In [ ]:
# Train with validation and checkpointing
result = train_tiny_model(
    cfg,
    train_tokens, train_labels,
    val_tokens=val_tokens, val_labels=val_labels,
    epochs=50,
    batch_size=64,
    lr=1e-3,
    checkpoint_every=10,
)

print(f"Final epoch: {result.final_epoch}")
print(f"Final train loss: {result.train_losses[-1]:.4f}")
print(f"Final val accuracy: {result.val_accs[-1]:.1%}")
print(f"Checkpoints at epochs: {sorted(result.checkpoints.keys())}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(result.train_losses)
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(result.val_losses, color="orange")
axes[1].set_title("Val Loss")
axes[1].set_xlabel("Epoch")

axes[2].plot(result.val_accs, color="green")
axes[2].set_title("Val Accuracy")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Accuracy")
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 3. Using the Trained Model

The trained model is a standard `HookedTransformer` — use `run_with_cache`, hooks, and all other
IRTK analysis tools on it.

In [ ]:
# Run a single example
test_a, test_b = 5, 8
test_tokens = jnp.array([test_a, test_b, 13])  # [a, b, =]
logits = result.model(test_tokens)

# Prediction at the last position
probs = jax.nn.softmax(logits[-1])
predicted = int(jnp.argmax(probs))
expected = (test_a + test_b) % 13
print(f"{test_a} + {test_b} mod 13 = {expected}")
print(f"Model predicted: {predicted} (prob: {float(probs[predicted]):.1%})")

In [ ]:
# Inspect attention patterns
_, cache = result.model.run_with_cache(test_tokens)
pattern = np.array(cache[("pattern", 0)])  # Layer 0 attention [n_heads, q, k]

fig, axes = plt.subplots(1, cfg.n_heads, figsize=(4 * cfg.n_heads, 3))
labels = [f"a={test_a}", f"b={test_b}", "="]
for h in range(cfg.n_heads):
    ax = axes[h] if cfg.n_heads > 1 else axes
    im = ax.imshow(pattern[h], cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_title(f"Head 0.{h}")
    plt.colorbar(im, ax=ax)

plt.suptitle("Attention Patterns for Modular Addition")
plt.tight_layout()
plt.show()

## 4. Checkpoint Analysis

Track how a metric evolves across training checkpoints.
This is useful for studying phase transitions and grokking.

In [ ]:
# Track max logit confidence at the last position across checkpoints
def confidence_metric(logits):
    """Max softmax probability at the last position."""
    probs = jax.nn.softmax(logits[-1])
    return float(jnp.max(probs))

test_seqs = [jnp.array([a, b, 13]) for a, b in [(3, 4), (7, 9), (1, 12)]]

analysis = training_checkpoint_analysis(result, test_seqs, confidence_metric)
print(f"Checkpoint epochs: {analysis['epochs']}")
print(f"Confidence scores: {analysis['metrics']}")

In [ ]:
# Track attention entropy across checkpoints
def attn_entropy_metric(logits):
    """This is a placeholder — in practice you'd cache and analyze attention."""
    probs = jax.nn.softmax(logits[-1])
    clipped = jnp.clip(probs, 1e-10, 1.0)
    return float(-jnp.sum(clipped * jnp.log(clipped)))

entropy_analysis = training_checkpoint_analysis(result, test_seqs, attn_entropy_metric)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(analysis["epochs"], analysis["metrics"], "o-")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Confidence")
axes[0].set_title("Prediction Confidence Across Training")

axes[1].plot(entropy_analysis["epochs"], entropy_analysis["metrics"], "o-", color="orange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Output Entropy")
axes[1].set_title("Output Entropy Across Training")

plt.tight_layout()
plt.show()

## 5. Different Tasks

You can train on other algorithmic tasks and compare.

In [ ]:
# Train on modular subtraction
sub_data = create_algorithmic_dataset("modular_subtraction", n_samples=400, modulus=7)
sub_cfg = irtk.HookedTransformerConfig(
    n_layers=1, d_model=32, n_ctx=4, d_head=16, n_heads=2, d_vocab=8,
)
sub_result = train_tiny_model(
    sub_cfg,
    sub_data["tokens"][:300], sub_data["labels"][:300],
    val_tokens=sub_data["tokens"][300:], val_labels=sub_data["labels"][300:],
    epochs=30, batch_size=64,
)

print(f"Subtraction final val accuracy: {sub_result.val_accs[-1]:.1%}")

# Test
a, b = 5, 3
logits = sub_result.model(jnp.array([a, b, 7]))
pred = int(jnp.argmax(logits[-1]))
print(f"{a} - {b} mod 7 = {(a-b)%7}, predicted: {pred}")

## Summary

| Function | Purpose |
|----------|--------|
| `create_algorithmic_dataset()` | Generate labeled data for modular arithmetic, copy, reverse, sort |
| `train_tiny_model()` | Train a HookedTransformer from scratch with AdamW |
| `training_checkpoint_analysis()` | Track a metric across training checkpoints |
| `TrainingResult` | Dataclass with model, losses, accuracies, and checkpoint snapshots |